# Jacobian lens on Gemma 4 E4B — text-only pilot

**Research question.** Does the Jacobian lens produce stable, interpretable, and
meaningfully transported vocabulary readouts from intermediate residual states
in `google/gemma-4-E4B-it`?

**Method source (authoritative):** [Verbalizable Representations Form a Global
Workspace in Language Models](https://transformer-circuits.pub/2026/workspace/index.html)
(Anthropic, 2026).
**Implementation scaffold:** the official reference implementation
[anthropics/jacobian-lens](https://github.com/anthropics/jacobian-lens)
(Apache-2.0), used here as the `upstream` git remote. This repository adapts it
to Gemma 4 with a narrow adapter (`jlens/gemma4.py`), controls
(`jlens/controls.py`), and metadata (`jlens/metadata.py`); the upstream
`jlens` core is unmodified.

**Jacobian convention** (paper §, upstream `jlens/fitting.py`):

$$J_\ell = \mathbb{E}_{\text{prompts},\,t,\,t' \ge t}\left[\frac{\partial h_{\text{final},t'}}{\partial h_{\ell,t}}\right],\qquad \text{lens}_\ell(h) = \mathrm{softmax}(W_U\,\mathrm{norm}(J_\ell h))$$

- `J[i, j] = ∂ h_final[dim i] / ∂ h_l[dim j]` — **rows are target dims, columns
  are source dims**; shape `[d_model, d_model] = [2560, 2560]`, fp32.
- Forward transport is `J @ h`, implemented as `residual @ J.T`
  (`JacobianLens.transport`). No other transposes appear in the pipeline.
- Estimator: one-hot cotangents at **every valid target position at once**;
  causal masking makes `t' < t` contributions exactly zero, so the gradient at
  source position `t` is the sum over `t' ≥ t`, then averaged over source
  positions (positions `0..15` and the final position are excluded) and prompts.
- Source site: output of `model.language_model.layers[l]` (after attention,
  MLP, per-layer-embedding re-injection, `layer_scalar`) — the input to block
  `l+1`. Target site: the same at block 41 (pre-final-norm residual).
- Readout: final RMSNorm → tied unembedding `[262144, 2560]` → **pre-softcap
  logits** (paper convention) and `30·tanh(x/30)` **softcapped logits**
  (Gemma's actual output pathway). The cap is monotonic ⇒ identical rankings.

Orientation is pinned by tests (`tests/test_fitting.py::test_jacobian_for_prompt_tiny`,
`tests/test_gemma4_adapter.py::test_fit_and_apply_through_adapter`,
`tests/test_finite_difference.py`). **Do not proceed if any of them fail.**


## 0. Colab setup (skip if running locally)

Running this notebook in a fresh Google Colab runtime requires a private-repo
checkout. Before running the next cell:

1. Create a **fine-grained GitHub personal access token** with **read-only**
   access to the `MechInterpreter/jacobian-lens-gemma` repository only.
2. In this Colab notebook, open the key icon in the left sidebar (**Secrets**)
   and add a new secret named `GITHUB_TOKEN` with the token as its value.
3. Toggle **Notebook access** on for that secret so this notebook may read it.

The next cell clones (or fast-forward-updates) `gemma4-e4b` into
`/content/jacobian-lens-gemma`, using the token only in memory for the git
operation — it is never printed, written to `.git/config`, or stored in the
remote URL. Running locally (outside Colab), this cell does nothing and the
existing checkout on disk is used instead.


In [ ]:
# 0. Colab bootstrap: clone/update the private repo from a fresh runtime.
# No-op outside Colab. Never loads Gemma; only touches git/pip.
import base64
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if not IN_COLAB:
    print("Not running in Colab \u2014 skipping bootstrap; using the local checkout.")
else:
    CHECKOUT_DIR = Path("/content/jacobian-lens-gemma")
    REPO_URL = "https://github.com/MechInterpreter/jacobian-lens-gemma.git"
    BRANCH = "gemma4-e4b"

    def _normalize(url: str) -> str:
        return url.strip().removesuffix(".git").removesuffix("/")

    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        GITHUB_TOKEN = None
    if not GITHUB_TOKEN:
        raise RuntimeError(
            "GITHUB_TOKEN secret not found or not accessible. In Colab: open "
            "the key icon (Secrets) in the left sidebar, add a secret named "
            "GITHUB_TOKEN containing a fine-grained GitHub token with "
            "read-only access to MechInterpreter/jacobian-lens-gemma, and "
            "enable notebook access for it. Then re-run this cell."
        )

    # Auth is passed via a per-invocation `-c http.<url>.extraHeader` config
    # override: it lives only in this process's argv, is never written to
    # .git/config, and never appears in the remote URL.
    _token_b64 = base64.b64encode(f"x-access-token:{GITHUB_TOKEN}".encode()).decode()
    _auth_header = f"AUTHORIZATION: basic {_token_b64}"
    _auth_args = ["-c", f"http.https://github.com/.extraHeader={_auth_header}"]

    def _run(args, *, auth=False, check=True):
        cmd = ["git", *(_auth_args if auth else []), *args]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if check and result.returncode != 0:
            safe_stderr = result.stderr.replace(GITHUB_TOKEN, "***")
            raise RuntimeError(f"git {' '.join(args)} failed:\n{safe_stderr}")
        return result.stdout.strip()

    if not CHECKOUT_DIR.exists():
        print(f"Cloning {REPO_URL} ({BRANCH}) into {CHECKOUT_DIR} ...")
        _run(["clone", "--branch", BRANCH, REPO_URL, str(CHECKOUT_DIR)], auth=True)
    else:
        if not (CHECKOUT_DIR / ".git").exists():
            raise RuntimeError(
                f"{CHECKOUT_DIR} exists but is not a git checkout; refusing to "
                "touch it. Remove or rename it manually, then re-run this cell."
            )
        existing_url = _run(["-C", str(CHECKOUT_DIR), "remote", "get-url", "origin"])
        if _normalize(existing_url) != _normalize(REPO_URL):
            raise RuntimeError(
                f"{CHECKOUT_DIR} is checked out from {existing_url!r}, not "
                f"{REPO_URL!r}; refusing to touch an unexpected repository. "
                "Remove or rename it manually, then re-run this cell."
            )
        print(f"Updating existing checkout at {CHECKOUT_DIR} ...")
        _run(["-C", str(CHECKOUT_DIR), "fetch", "origin"], auth=True)
        _run(["-C", str(CHECKOUT_DIR), "checkout", BRANCH])
        _run(["-C", str(CHECKOUT_DIR), "merge", "--ff-only", f"origin/{BRANCH}"])

    # Defensive: the origin URL must stay credential-free no matter what.
    final_url = _run(["-C", str(CHECKOUT_DIR), "remote", "get-url", "origin"])
    if "@" in final_url or GITHUB_TOKEN in final_url:
        raise RuntimeError("origin URL unexpectedly contains credentials; aborting.")

    os.chdir(CHECKOUT_DIR)
    if str(CHECKOUT_DIR) not in sys.path:
        sys.path.insert(0, str(CHECKOUT_DIR))

    print("Installing the 'gemma' extra ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[gemma]"],
        check=True,
    )

    branch_now = _run(["-C", str(CHECKOUT_DIR), "rev-parse", "--abbrev-ref", "HEAD"])
    sha_now = _run(["-C", str(CHECKOUT_DIR), "rev-parse", "HEAD"])
    print(f"checked out: {branch_now} @ {sha_now}")
    print("remotes:")
    print(_run(["-C", str(CHECKOUT_DIR), "remote", "-v"]))

    assert (CHECKOUT_DIR / "jlens").is_dir(), f"{CHECKOUT_DIR}/jlens not found"
    import importlib
    importlib.invalidate_caches()
    import jlens as _jlens_probe
    resolved = Path(_jlens_probe.__file__).resolve().parent
    expected = (CHECKOUT_DIR / "jlens").resolve()
    assert resolved == expected, f"jlens resolved from {resolved}, expected {expected}"
    print(f"jlens resolves from: {resolved}")
    del _jlens_probe


In [ ]:
# 1. Environment, upstream commit, repository provenance (no model load).
import json, os, sys, pathlib

REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / "jlens").exists():          # notebook started inside notebooks/
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from jlens.metadata import environment_manifest, UPSTREAM_COMMIT, UPSTREAM_REPO_URL

ENV = environment_manifest()
print(f"upstream: {UPSTREAM_REPO_URL} @ {UPSTREAM_COMMIT}")
print(json.dumps(ENV, indent=2))
IN_COLAB = "google.colab" in sys.modules
print("IN_COLAB:", IN_COLAB)


## Persisting outputs to Google Drive

The repository checkout at `/content/jacobian-lens-gemma` stays **ephemeral**
— it is git-tracked and reproducible, so nothing is lost if it disappears on a
runtime reset. Everything a reset *would* otherwise destroy (fitted lens
artifacts, fitting checkpoints, experiment metadata, control results,
human-readable summaries, and a per-execution run manifest) is instead
persisted to **Google Drive**, under:

`My Drive/jacobian-lens-gemma/`
- `artifacts/<mode>/` — `lens.pt`, `fit_metadata.json`, `eval_metadata.json`,
  the fitting checkpoint (mirrors the existing local `artifacts/<mode>/`
  layout, just rooted on Drive instead of `/content`)
- `checkpoints/` — reserved persistent root for checkpoint storage
- `runs/<run_id>/` — one directory per execution: `run_metadata.json` (config,
  fingerprint, resolved Gemma revision, local commit, prompt hashes,
  architecture report, runtime/peak-memory, logit-lens and J-lens readouts,
  control results, and paths to the artifacts above) plus a human-readable
  `summary.md`

Running the next cell in Colab prompts you to **authorize Google Drive
access** — approve it in the popup that appears. **Model weights are never
copied to Drive**; the Hugging Face cache stays under `/content` so model
loading remains fast. Re-running this notebook (or any fitting/evaluation
cell) creates a **new** run directory under `runs/` rather than overwriting a
prior one. Outside Colab this section is a no-op: outputs stay in the
repository's local `artifacts/` directories exactly as before.


In [ ]:
# Google Drive persistence (Colab only): mount Drive and resolve the
# persistent root. Runs before any model loading, fitting, checkpointing, or
# artifact creation. No-op outside Colab \u2014 local relative paths under
# artifacts/ are used unchanged and no Drive mount is required.
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        raise RuntimeError(
            f"failed to mount Google Drive at /content/drive: {exc}. Approve "
            "the Drive authorization prompt when it appears, then re-run "
            "this cell."
        ) from exc

    DRIVE_MOUNT = Path("/content/drive")
    if not DRIVE_MOUNT.is_dir():
        raise RuntimeError(
            f"{DRIVE_MOUNT} does not exist after drive.mount() returned; "
            "Drive did not mount successfully."
        )

    PERSIST_ROOT = DRIVE_MOUNT / "MyDrive" / "jacobian-lens-gemma"
    ARTIFACTS_ROOT = PERSIST_ROOT / "artifacts"
    CHECKPOINTS_ROOT = PERSIST_ROOT / "checkpoints"
    RUNS_ROOT = PERSIST_ROOT / "runs"
    try:
        for _d in (PERSIST_ROOT, ARTIFACTS_ROOT, CHECKPOINTS_ROOT, RUNS_ROOT):
            _d.mkdir(parents=True, exist_ok=True)
    except OSError as exc:
        raise RuntimeError(
            f"failed to create persistent directories under {PERSIST_ROOT}: {exc}"
        ) from exc

    print(f"repository checkout (ephemeral): {CHECKOUT_DIR}")
    print("Google Drive persistent root (survives a runtime reset):")
    print(f"  PERSIST_ROOT      = {PERSIST_ROOT}")
    print(f"  ARTIFACTS_ROOT    = {ARTIFACTS_ROOT}")
    print(f"  CHECKPOINTS_ROOT  = {CHECKPOINTS_ROOT}")
    print(f"  RUNS_ROOT         = {RUNS_ROOT}")
else:
    PERSIST_ROOT = ARTIFACTS_ROOT = CHECKPOINTS_ROOT = RUNS_ROOT = None
    print("Not running in Colab \u2014 outputs stay local under the repository's "
          "relative artifacts/ paths (unchanged behaviour); Drive is not used.")


In [ ]:
# OPTIONAL recovery: if this Colab runtime already completed a micro-smoke
# fit under the old (pre-Drive-persistence) ephemeral layout, copy its
# outputs into a fresh Drive run directory before they are lost to a reset.
# Never re-fits, never loads Gemma, never deletes or overwrites anything.
# Skipped by default; set RUN_RECOVERY = True and re-run this cell to act.
RUN_RECOVERY = False

if not RUN_RECOVERY:
    print("recovery cell is a no-op (RUN_RECOVERY=False); set it to True and "
          "re-run this cell to copy an already-completed local micro-smoke "
          "into Drive")
else:
    if not IN_COLAB or PERSIST_ROOT is None:
        raise RuntimeError(
            "recovery requires a mounted Drive persistent root; run the "
            "Drive persistence cell above first"
        )
    import shutil
    from datetime import datetime, timezone

    _local_output_dir = Path("artifacts/microsmoke")
    _local_checkpoint = Path("artifacts/microsmoke/ckpt.pt")
    _found = [p for p in (_local_output_dir, _local_checkpoint) if p.exists()]
    if not _found:
        print(f"nothing found at {_local_output_dir} or {_local_checkpoint}; "
              "nothing to recover")
    else:
        _stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")
        _recovery_dir = RUNS_ROOT / f"recovered_microsmoke_{_stamp}"
        if _recovery_dir.exists():
            raise RuntimeError(f"{_recovery_dir} already exists; refusing to overwrite")
        _recovery_dir.mkdir(parents=True)
        _copied = []
        if _local_output_dir.exists():
            _dest = _recovery_dir / "artifacts"
            shutil.copytree(_local_output_dir, _dest, copy_function=shutil.copy2)
            _copied.append((_local_output_dir, _dest))
        if _local_checkpoint.exists() and not _local_checkpoint.resolve().is_relative_to(
            _local_output_dir.resolve()
        ):
            _dest = _recovery_dir / "checkpoints" / _local_checkpoint.name
            _dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(_local_checkpoint, _dest)
            _copied.append((_local_checkpoint, _dest))
        print(f"recovered into {_recovery_dir}:")
        for _src, _dst in _copied:
            print(f"  {_src}  ->  {_dst}")


### Configuration and gating

`MODE` selects `configs/gemma_text_{microsmoke,smoke,pilot}.yaml`.
Real Gemma execution (~16 GB download + load) happens **only** if
`ALLOW_MODEL_LOAD` is true — set the environment variable `JLENS_ALLOW_GEMMA=1`
(or flip the constant) deliberately. On Colab use an **A100** runtime; begin
with the config's small `dim_batch` and scale up only after the memory probe.
Stage order: `microsmoke` (1–2 prompts, one middle layer, seq ≤ 48,
dim_batch 4) → `smoke` (8 prompts, 5 layers) → `pilot` (100 WikiText
sequences, 7 layers). Smoke-mode lens quality does **not** represent the
method; only the pilot is worth interpreting.


In [ ]:
# 2. Load and validate the experiment configuration.
from datetime import datetime, timezone

from jlens.metadata import load_config, config_fingerprint

MODE = os.environ.get("JLENS_MODE", "microsmoke")            # microsmoke | smoke | pilot
ALLOW_MODEL_LOAD = os.environ.get("JLENS_ALLOW_GEMMA", "0") == "1"
DEVICE_MAP = os.environ.get("JLENS_DEVICE_MAP") or None       # e.g. "cuda" on the A100

CONFIG_PATH = f"configs/gemma_text_{MODE}.yaml"
config = load_config(CONFIG_PATH)
FINGERPRINT = config_fingerprint(config)
print(f"config: {CONFIG_PATH}\nfingerprint: {FINGERPRINT}")
print(f"mode={config['mode']}  source_layers={config['sites']['source_layers']}  "
      f"target_layer={config['sites']['target_layer']}")
print(f"ALLOW_MODEL_LOAD={ALLOW_MODEL_LOAD}  DEVICE_MAP={DEVICE_MAP}")

# Resolve output/checkpoint paths against the Drive persistent root in Colab;
# outside Colab (PERSIST_ROOT is None) the existing repo-relative behaviour
# is unchanged.
if PERSIST_ROOT is not None:
    OUTPUT_DIR = PERSIST_ROOT / config["paths"]["output_dir"]
    CHECKPOINT_PATH = PERSIST_ROOT / config["paths"]["checkpoint"]
else:
    OUTPUT_DIR = pathlib.Path(config["paths"]["output_dir"])
    CHECKPOINT_PATH = pathlib.Path(config["paths"]["checkpoint"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

# A distinct, auditable run directory per execution (Colab + Drive only) —
# timestamp + fingerprint so a rerun never silently overwrites a prior run.
if RUNS_ROOT is not None:
    RUN_ID = (f"{MODE}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')}_"
              f"{FINGERPRINT.removeprefix('sha256:')[:12]}")
    RUN_DIR = RUNS_ROOT / RUN_ID
    RUN_DIR.mkdir(parents=True, exist_ok=False)
else:
    RUN_ID = RUN_DIR = None

print(f"OUTPUT_DIR = {OUTPUT_DIR}")
print(f"CHECKPOINT_PATH = {CHECKPOINT_PATH}")
print(f"RUN_DIR = {RUN_DIR}")


In [ ]:
# 3. Lightweight validation suite (CPU, mocks, no network) — must pass before
# any real-model work.
import subprocess
proc = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "--no-header"],
                      capture_output=True, text=True, cwd=REPO_ROOT)
print(proc.stdout[-2000:])
assert proc.returncode == 0, "local validation suite failed — do not proceed"


In [ ]:
# 4. Resolve the immutable model revision (network, but no model download).
from jlens.gemma4 import resolve_revision

MODEL_REVISION = None
if ALLOW_MODEL_LOAD:
    MODEL_REVISION = resolve_revision(config["model"]["repo_id"],
                                      config["model"]["revision"])
    print(f"{config['model']['repo_id']} pinned to {MODEL_REVISION}")
else:
    print("model load disabled — skipping revision resolution")


In [ ]:
# 5. Load Gemma 4 (gated) and verify the architecture.
model = None
if ALLOW_MODEL_LOAD:
    import torch
    from jlens.gemma4 import load_gemma4, verify_architecture

    dtype = {"bfloat16": torch.bfloat16, "float32": torch.float32}[config["model"]["dtype"]]
    model, LOAD_INFO = load_gemma4(
        config["model"]["repo_id"], revision=MODEL_REVISION, dtype=dtype,
        device_map=DEVICE_MAP, allow_model_load=True)
    ARCH_REPORT = verify_architecture(
        model,
        expect_n_layers=config["model"]["expect_n_layers"],
        expect_d_model=config["model"]["expect_d_model"],
        expect_vocab_size=config["model"]["expect_vocab_size"])
    print(json.dumps({**LOAD_INFO, **{k: v for k, v in ARCH_REPORT.to_dict().items()
                                      if k != "layer_scalars"}}, indent=2, default=str))
    print("layer_scalars all unit:", ARCH_REPORT.layer_scalars_all_unit)
else:
    print("model load disabled — skipping (set JLENS_ALLOW_GEMMA=1)")


In [ ]:
# 6. Hook-site resolution + memory/runtime probe at the configured dim_batch.
# Scale dim_batch upward ONLY after inspecting the probe's peak memory.
PROBE = None
if model is not None:
    from jlens.gemma4 import probe_fit_cost
    import json as _json

    print("source blocks:", [f"model.language_model.layers[{l}]"
                             for l in config["sites"]["source_layers"]])
    print(f"target block:  model.language_model.layers[{config['sites']['target_layer']}]")
    fit_prompts_preview = json.load(open(config["fitting"]["prompts_path"] or
                                         "configs/prompts/fit_prompts.json",
                                         encoding="utf-8"))["prompts"]
    PROBE = probe_fit_cost(model, fit_prompts_preview[0],
                           config["sites"]["source_layers"],
                           dim_batch=config["fitting"]["dim_batch"],
                           max_seq_len=min(48, config["fitting"]["max_seq_len"]))
    print(_json.dumps(PROBE, indent=2))
    assert PROBE["all_finite"]


In [ ]:
# 7. Fit the lens (checkpointed, resumable) — or load an existing artifact.
LENS = None
FIT_RUNTIME_SECONDS = None
PROMPT_HASHES = None
if model is not None:
    import time
    from jlens.fitting import fit
    from jlens.lens import JacobianLens
    from jlens.metadata import environment_manifest, prompt_hashes, write_metadata

    lens_path = OUTPUT_DIR / "lens.pt"
    if lens_path.exists():
        LENS = JacobianLens.load(str(lens_path))
        print("loaded existing", LENS)
    else:
        if config["fitting"]["prompt_source"] == "file":
            prompts = json.load(open(config["fitting"]["prompts_path"],
                                     encoding="utf-8"))["prompts"][:config["fitting"]["n_prompts"]]
        else:
            from jlens.examples import load_wikitext_prompts
            prompts = load_wikitext_prompts(n_prompts=config["fitting"]["n_prompts"])
        start = time.perf_counter()
        LENS = fit(model, prompts,
                   source_layers=config["sites"]["source_layers"],
                   target_layer=config["sites"]["target_layer"],
                   dim_batch=config["fitting"]["dim_batch"],
                   max_seq_len=config["fitting"]["max_seq_len"],
                   skip_first=config["positions"]["skip_first"],
                   checkpoint_path=str(CHECKPOINT_PATH),
                   checkpoint_every=config["fitting"]["checkpoint_every"])
        FIT_RUNTIME_SECONDS = time.perf_counter() - start
        LENS.save(str(lens_path))
        assert JacobianLens.load(str(lens_path)).source_layers == LENS.source_layers
        PROMPT_HASHES = prompt_hashes(prompts)
        write_metadata(str(OUTPUT_DIR / "fit_metadata.json"), {
            "config": config, "config_fingerprint": FINGERPRINT,
            "load_info": LOAD_INFO, "architecture_report": ARCH_REPORT.to_dict(),
            "probe": PROBE, "prompt_hashes": PROMPT_HASHES,
            "n_prompts_fitted": LENS.n_prompts,
            "fit_runtime_seconds": round(FIT_RUNTIME_SECONDS, 1),
            "environment": environment_manifest()})
        print(f"fitted + saved {lens_path} in {FIT_RUNTIME_SECONDS:.0f}s:", LENS)


In [ ]:
# 8. Logit lens vs J-lens on the evaluation prompts.
# Plain-text and chat-templated prompts are evaluated SEPARATELY; the fitting
# corpus was plain text only. Both pre-softcap (paper convention) and
# softcapped (Gemma pathway) logit values are shown; rankings are identical.
EVAL_READOUTS = []
if LENS is not None:
    from jlens.gemma4 import apply_dual

    eval_payload = json.load(open(config["eval"]["prompts_path"], encoding="utf-8"))
    K = config["eval"]["top_k"]

    def show(text, slug, fmt, positions):
        lens_logits, model_logits, ids = apply_dual(LENS, model, text, positions=positions)
        logit_logits, _, _ = apply_dual(LENS, model, text, positions=positions,
                                        use_jacobian=False)
        print(f"\n=== {slug} [{fmt}] (seq_len={ids.shape[1]}) ===")
        for i, pos in enumerate(positions):
            actual = model.tokenizer.decode([int(model_logits["pre"][i].argmax())])
            print(f" position {pos} — model top-1: {actual!r}")
            row = {"slug": slug, "format": fmt, "position": pos,
                   "model_top1": actual, "layers": {}}
            for layer in LENS.source_layers:
                jl = lens_logits[layer]["pre"][i]; jc = lens_logits[layer]["capped"][i]
                ll = logit_logits[layer]["pre"][i]
                fmt_top = lambda t, n=5: " ".join(repr(model.tokenizer.decode([int(x)]))
                                                  for x in t.topk(n).indices)
                print(f"  L{layer:>2}  J-lens: {fmt_top(jl)}")
                print(f"       (top-1 logit pre={jl.max():.2f} capped={jc.max():.2f})")
                print(f"       logit-lens: {fmt_top(ll)}")
                row["layers"][layer] = {
                    "jlens_top5": [model.tokenizer.decode([int(x)]) for x in jl.topk(5).indices],
                    "jlens_top1_logit_pre": float(jl.max()),
                    "jlens_top1_logit_capped": float(jc.max()),
                    "logit_lens_top5": [model.tokenizer.decode([int(x)]) for x in ll.topk(5).indices],
                }
            EVAL_READOUTS.append(row)

    for entry in eval_payload["plain"]:
        show(entry["text"], entry["slug"], "plain", entry.get("positions", [-1]))
    for entry in eval_payload["chat"]:
        messages = ([{"role": "system", "content": entry["system"]}] if entry.get("system") else [])
        messages.append({"role": "user", "content": entry["user"]})
        text = model.tokenizer.apply_chat_template(messages, tokenize=False,
                                                   add_generation_prompt=True)
        show(text, entry["slug"], "chat", entry.get("positions", [-1]))


In [ ]:
# 9. Negative controls. Primary: row-permuted fitted J (same entries, transport
# destroyed). Also: scale-matched random matrix, wrong-layer application, and
# the identity/logit lens. Metrics vs the model's real output distribution.
if LENS is not None:
    from jlens.controls import control_lens, wrong_layer_lens, topk_overlap, ranks_of_targets

    controls = {"permuted": control_lens(LENS, "permuted", seed=config["eval"]["control_seed"]),
                "random": control_lens(LENS, "random", seed=config["eval"]["control_seed"])}
    if len(LENS.source_layers) >= 2:
        controls["wrong_layer"] = wrong_layer_lens(LENS)

    text = eval_payload["plain"][0]["text"]; positions = [-2, -1]
    jl, mdl, _ = apply_dual(LENS, model, text, positions=positions)
    ll, _, _ = apply_dual(LENS, model, text, positions=positions, use_jacobian=False)
    model_top1 = mdl["pre"].argmax(-1)
    CONTROL_ROWS = []
    print(f"top-{config['eval']['top_k']} overlap with model output / rank of model top-1")
    print(f"{'layer':>6} {'J-lens':>14} {'logit-lens':>14} {'permuted':>14} {'random':>14} {'wrong-layer':>14}")
    for layer in LENS.source_layers:
        row = {"layer": layer}
        variants = {"J-lens": jl[layer]["pre"], "logit-lens": ll[layer]["pre"]}
        for name, c in controls.items():
            variants[name] = c.apply(model, text, layers=[layer], positions=positions)[0][layer].float()
        cols = []
        for name, logits in variants.items():
            ov = topk_overlap(logits, mdl["pre"], config["eval"]["top_k"])
            rk = int(ranks_of_targets(logits, model_top1)[-1])
            row[name] = {"overlap": round(ov, 3), "rank_of_model_top1": rk}
            cols.append(f"{ov:.2f} / r{rk:>5}")
        CONTROL_ROWS.append(row)
        print(f"{layer:>6} " + " ".join(f"{c:>14}" for c in cols))
    print("\nMeaningful transport ⇒ J-lens should beat permuted/random/wrong-layer,"
          "\nespecially at early/middle layers where the logit lens also degrades.")


In [ ]:
# 10. Optional: upstream slice visualisation (layer × position heatmap).
# Reused as-is from upstream jlens.vis; skipped gracefully if incompatible.
if LENS is not None:
    try:
        from jlens.vis import build_page, compute_slice, notebook_iframe
        slice_prompt = eval_payload["plain"][0]["text"]
        slice_data = compute_slice(model, LENS, slice_prompt)
        page, _, _ = build_page(slice_data, slice_prompt,
                                title="Gemma 4 E4B — multihop-currency",
                                description="J-lens slice (upstream visualisation)",
                                mode="embed")
        out = OUTPUT_DIR / "slice_multihop.html"
        out.write_text(page, encoding="utf-8")
        print("wrote", out)
        display(notebook_iframe(page))
    except Exception as exc:                      # noqa: BLE001
        print(f"slice visualisation skipped: {type(exc).__name__}: {exc}")


In [ ]:
# 11. Save the evaluation + control artifacts (configuration-bound), plus a
# per-execution run manifest and human-readable summary under Drive (Colab).
if LENS is not None:
    from jlens.metadata import write_metadata, environment_manifest
    write_metadata(str(OUTPUT_DIR / "eval_metadata.json"), {
        "config": config, "config_fingerprint": FINGERPRINT,
        "model_revision": MODEL_REVISION,
        "eval_readouts": EVAL_READOUTS,
        "control_rows": CONTROL_ROWS,
        "notes": ("pre-softcap logits follow the paper's lens definition; "
                  "softcapped logits follow Gemma's output pathway; rankings "
                  "identical (monotonic cap). Chat prompts evaluated separately "
                  "from the plain-text fitting distribution."),
        "environment": environment_manifest()})
    print("wrote", OUTPUT_DIR / "eval_metadata.json")
    print(sorted(p.name for p in OUTPUT_DIR.iterdir()))

    if RUN_DIR is not None:
        # Auditable per-execution manifest: references the artifacts above
        # (via jlens.metadata.write_metadata, the project's existing
        # mechanism) rather than duplicating them.
        write_metadata(str(RUN_DIR / "run_metadata.json"), {
            "run_id": RUN_ID,
            "mode": MODE,
            "config": config,
            "config_fingerprint": FINGERPRINT,
            "model_revision": MODEL_REVISION,
            "load_info": LOAD_INFO,
            "architecture_report": ARCH_REPORT.to_dict(),
            "probe": PROBE,
            "fit_runtime_seconds": (
                round(FIT_RUNTIME_SECONDS, 1) if FIT_RUNTIME_SECONDS is not None else None
            ),
            "prompt_hashes": PROMPT_HASHES,
            "eval_readouts": EVAL_READOUTS,
            "control_rows": CONTROL_ROWS,
            "artifact_paths": {
                "lens": str(OUTPUT_DIR / "lens.pt"),
                "checkpoint": str(CHECKPOINT_PATH),
                "fit_metadata": str(OUTPUT_DIR / "fit_metadata.json"),
                "eval_metadata": str(OUTPUT_DIR / "eval_metadata.json"),
            },
            "environment": environment_manifest(),
        })

        summary_lines = [
            f"# Run {RUN_ID}",
            "",
            f"- mode: {MODE}",
            f"- config: {CONFIG_PATH} (fingerprint {FINGERPRINT})",
            f"- model: {config['model']['repo_id']} @ {MODEL_REVISION}",
            f"- lens: {OUTPUT_DIR / 'lens.pt'} ({LENS.n_prompts} prompts fitted)",
            f"- checkpoint: {CHECKPOINT_PATH}",
            "",
            "## Controls (top-k overlap with model output / rank of model top-1)",
            "",
        ]
        for row in CONTROL_ROWS:
            variants = ", ".join(
                f"{name}={vals['overlap']:.2f}/r{vals['rank_of_model_top1']}"
                for name, vals in row.items() if name != "layer"
            )
            summary_lines.append(f"- layer {row['layer']}: {variants}")
        (RUN_DIR / "summary.md").write_text("\n".join(summary_lines), encoding="utf-8")

        print(f"wrote run manifest: {RUN_DIR / 'run_metadata.json'}")
        print(f"wrote run summary:  {RUN_DIR / 'summary.md'}")


## Summary and limitations

Fill in after a real run. Template:

- **Stage reached:** microsmoke / smoke / pilot; model revision `…`; runtime, peak memory.
- **Feasibility:** did fitting run end-to-end with finite Jacobians and stable
  per-prompt norms (`fit()` logs `max||J||/sqrt(d)` and running-mean shift)?
- **Interpretability:** at which layers does the J-lens read out
  contextually sensible tokens where the logit lens does not?
- **Controls:** J-lens vs permuted / random / wrong-layer overlap and ranks —
  is apparent interpretability attributable to learned transport?
- **Known limitations:** smoke-scale corpus (≪ paper's ~1000 sequences); bf16
  backward noise; sliding-window + KV-shared attention differ from the paper's
  models; text-only; `-it` chat distribution vs plain-text fitting corpus.

**Extension points (deliberately not implemented):** sparse J-space
decomposition, gradient pursuit, k-cone discovery/visualisation, steering,
multimodal inputs — see README § Roadmap.
